# Testing Tiled integration through Tango

This notebook checks the real-hardware Tiled flow:

1. Connect to Tango devices.
2. Configure the Tiled Tango device.
3. Acquire a STEM image from ThermoMicroscope.
4. Use the returned descriptor to retrieve data through the Tiled device.


### Quick Start Code Cell

In [1]:
import subprocess
import os
import json
import time
from getpass import getpass

import tango
import numpy as np
import matplotlib.pyplot as plt


In [12]:
# Tango / Tiled configuration
DB_HOST = "localhost"
DB_PORT = 9000

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

SCAN_DEVICE = "test/scan/1"
MICROSCOPE_DEVICE = "test/microscope/1"
TILED_DEVICE = "test/tiled/1"

TILED_HOST = "10.46.217.241"
TILED_PORT = 9091
SAVE_PATH = "D:/microscopedata/ahoust17/20260511/"  # Change this to the directory watched by Tiled
TILED_ROOT_PATH = ""  # Optional path prefix inside Tiled

print("TANGO_HOST =", os.environ["TANGO_HOST"])


TANGO_HOST = localhost:9000


## 0. Start local Tango servers


In [13]:
# ── Config ────────────────────────────────────────────────────────────────────
DB_HOST = "localhost"
DB_PORT = 9000
server_list = [("stage", "asyncroscopy.hardware.STAGE"),
               ("scan", "asyncroscopy.hardware.SCAN"),
               ("eds", "asyncroscopy.detectors.EDS"),
               ("camera", "asyncroscopy.detectors.CAMERA"),
               ("tiled", "asyncroscopy.Tiled")]
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = os.path.dirname(os.getcwd())
os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"
env = {**os.environ, "TANGO_HOST": f"{DB_HOST}:{DB_PORT}"}
processes = {}

def popen(cmd):
    return subprocess.Popen(cmd, env=env, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def wait_for_device(name, timeout=60, interval=1):
    print(f"  Waiting for {name}...", end="", flush=True)
    start = time.time()
    while time.time() - start < timeout:
        try:
            tango.DeviceProxy(name).ping()
            print(f" ✅ ready ({time.time()-start:.1f}s)")
            return True
        except Exception:
            print(".", end="", flush=True)
            time.sleep(interval)
    print(f" ❌ timed out after {timeout}s")
    return False

def check_processes(*names):
    for name in names:
        p = processes[name]
        print(f"\n─── {name} (PID {p.pid}) ───\n  Running: {p.poll() is None}")
        for label, fd in [("STDOUT", p.stdout), ("STDERR", p.stderr)]:
            try:
                print(f"  {label}: {fd.read1(4096).decode() or '(empty)'}")
            except Exception:
                print(f"  {label}: (no output yet)")

# Kill old processes (if any)
print("Clearing old processes...")
for cmd in [f"kill -9 $(lsof -t -i:{DB_PORT}) 2>/dev/null || true",
            *[f"pkill -f '{module.split('.')[-1]} {name}_instance' 2>/dev/null || true"
              for name, module in server_list],
            "pkill -f 'ThermoMicroscope microscope_instance' 2>/dev/null || true"]:
    subprocess.run(cmd, shell=True)
time.sleep(2)

# Start DB
print(f"Project dir: {PROJECT_DIR}\nStarting Tango Database...")
processes["database"] = popen(["uv", "run", "python", "-m", "tango.databaseds.database", "2"])
print("  Waiting for database...", end="", flush=True)
for _ in range(30):
    try:
        tango.Database(DB_HOST, DB_PORT)
        print(" ✅ ready")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(1)
else:
    check_processes("database")
    raise RuntimeError("Tango database failed to start.")

# Register devices
print("Registering devices...")
r = subprocess.run(["uv", "run", "scripts/2_register_devices.py"], env=env, cwd=PROJECT_DIR, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print("ERROR:", r.stderr)
    raise RuntimeError("Device registration failed — stopping here.")

# Start servers
print("Starting device servers...")
for name, module in server_list:
    processes[name] = popen(["uv", "run", "python", "-m", module, f"{name}_instance"])

if not all(wait_for_device(f"test/{d}/1") for d in ["stage", "scan", "eds", "camera", "tiled"]):
    print("\n⚠️  Debug info:")
    check_processes("stage", "scan", "eds", "camera", "tiled")
    raise RuntimeError("One or more device servers failed.")

print("Starting Microscope...")
processes["microscope"] = popen(["uv", "run", "python", "-m", "asyncroscopy.ThermoMicroscope", "microscope_instance"])
if not wait_for_device("test/microscope/1"):
    print("\n⚠️  Debug info:")
    check_processes("microscope")
    raise RuntimeError("Microscope server failed — see debug info above.")

print("\n✅ All servers ready!")


Clearing old processes...
Project dir: /Users/austin/Documents/GitHub/asyncroscopy
Starting Tango Database...
  Waiting for database.... ✅ ready
Registering devices...
Connected: stingray-zm68g.device.utk.edu:9000

  registered: test/scan/1
  registered: test/camera/1
  registered: test/eds/1
  registered: test/stage/1
  registered: test/corrector/1
  registered: test/tiled/1
  registered: test/microscope/1
  property:   scan_device_address = test/scan/1
  property:   camera_device_address = test/camera/1
  property:   eds_device_address = test/eds/1
  property:   stage_device_address = test/stage/1
  property:   corrector_device_address = test/corrector/1
  property:   tiled_device_address = test/tiled/1

Done!
Starting device servers...
  Waiting for test/stage/1.... ✅ ready (1.0s)
  Waiting for test/scan/1... ✅ ready (0.0s)
  Waiting for test/eds/1... ✅ ready (0.0s)
  Waiting for test/camera/1... ✅ ready (0.0s)
  Waiting for test/tiled/1... ✅ ready (0.0s)
Starting Microscope...
  Wa

## 1. Connect to devices

In [14]:
db = tango.Database()
print("Devices registered in Tango DB:\n")
for device in db.get_device_name("*", "*"):
    print(device)


Devices registered in Tango DB:

asyncroscopy/detector/haadf
asyncroscopy/microscope/thermo
dserver/CAMERA/camera_instance
dserver/CORRECTOR/corrector_instance
dserver/DataBaseds/2
dserver/DetectorServer/detectors
dserver/EDS/eds_instance
dserver/HAADF/haadf_instance
dserver/MicroscopeServer/microscope
dserver/SCAN/scan_instance
dserver/STAGE/eds_instance
dserver/STAGE/stage_instance
dserver/TangoAccessControl/1
dserver/TangoTest/test
dserver/ThermoMicroscope/microscope_instance
dserver/Tiled/tiled_instance
sys/access_control/1
sys/database/2
sys/tg_test/1
test/camera/1
test/corrector/1
test/eds/1
test/microscope/1
test/scan/1
test/stage/1
test/tiled/1


In [15]:
scan = tango.DeviceProxy(SCAN_DEVICE)
microscope = tango.DeviceProxy(MICROSCOPE_DEVICE)
tiled = tango.DeviceProxy(TILED_DEVICE)

for proxy in (scan, microscope, tiled):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())
print("tiled     :", tiled.state())


scan      : ON
microscope: ON
tiled     : ON


## 2. Configure the Tiled Tango device

In [16]:
tiled.host = TILED_HOST
tiled.port = TILED_PORT
tiled.save_path = SAVE_PATH
tiled.root_path = TILED_ROOT_PATH

# The API key must be set on the Tango device process, not only in this notebook.
tiled.set_api_key(getpass("Enter your Tiled API key: "))

print(json.dumps(json.loads(tiled.get_config()), indent=2))


{
  "host": "10.46.217.241",
  "port": 9091,
  "uri": "http://10.46.217.241:9091",
  "save_path": "C:/AE_future/ahoust17/20260511/",
  "root_path": "",
  "api_key_configured": true
}


In [17]:
# If ThermoMicroscope was registered with tiled_device_address, this should reflect the Tiled device settings.
# If it does not, update the Tango DB property tiled_device_address = test/tiled/1 and restart ThermoMicroscope.
try:
    print(json.dumps(json.loads(microscope.get_tiled_acquisition_config()), indent=2))
except Exception as exc:
    print("Could not read microscope tiled config:", exc)


{
  "save_directory": "C:/AE_future/ahoust17/20260511/",
  "tiled_uri": "http://10.46.217.241:9091",
  "tiled_root_path": "",
  "file_format": "tiff"
}


## 3. Configure a small STEM acquisition

In [18]:
scan.Activate(["haadf"])
scan.dwell_time = 1e-6
scan.imsize = 128

print("dwell_time:", scan.dwell_time)
print("imsize    :", scan.imsize)


dwell_time: 1e-06
imsize    : 128


## 4. Acquire and inspect the descriptor

In [19]:
descriptor_json = microscope.get_scanned_image()
descriptor = json.loads(descriptor_json)

print(json.dumps(descriptor, indent=2))
print("\nSaved file:", descriptor.get("path"))
print("Format    :", descriptor.get("format"))
print("Exists    :", descriptor.get("file_exists"))
print("Bytes     :", descriptor.get("file_size_bytes"))
print("Candidates:")
for candidate in descriptor.get("tiled_path_candidates", []):
    print("  -", candidate)

# Ask the Tiled Tango device process what it can see on disk.
try:
    print("\nTiled device path check:")
    print(json.dumps(json.loads(tiled.path_exists(descriptor["path"])), indent=2))
except Exception as exc:
    print("Could not run tiled.path_exists:", exc)


DevFailed: DevFailed[
    DevError[
        desc = FileNotFoundError: [Errno 2] No such file or directory: 'C:/AE_future/ahoust17/20260511/stem_image_haadf_20260511T160310_0be8d785.tiff'
        origin = Traceback (most recent call last):
              File "/Users/austin/Documents/GitHub/asyncroscopy/.venv/lib/python3.12/site-packages/tango/server.py", line 1790, in wrapped_command_method
                return get_worker().execute(cmd_method, *args, **kwargs)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              File "/Users/austin/Documents/GitHub/asyncroscopy/.venv/lib/python3.12/site-packages/tango/green.py", line 110, in execute
                return fn(*args, **kwargs)
                       ^^^^^^^^^^^^^^^^^^^
              File "/Users/austin/Documents/GitHub/asyncroscopy/asyncroscopy/ThermoMicroscope.py", line 198, in get_scanned_image
                return self._acquire_stem_image(imsize, dwell_time, ["haadf"])
                       ^^^^^^

## 5. Check recent files from the Tiled device

In [17]:
print("Tiled root entries:")
try:
    print(json.dumps(json.loads(tiled.list_root()), indent=2)[:4000])
except Exception as exc:
    print("Could not list Tiled root:", exc)

recent = json.loads(tiled.get_recent())
print("\nRecent files visible to the Tiled Tango device process:")
print(json.dumps(recent, indent=2)[:4000])


{
  "save_path": "D:/microscopedata/ahoust17/20260511/",
  "files": [
    {
      "path": "D:/microscopedata/ahoust17/20260511/stem_image_haadf_20260511T154531_548f5184.tiff",
      "file_name": "stem_image_haadf_20260511T154531_548f5184.tiff",
      "relative_path": "stem_image_haadf_20260511T154531_548f5184.tiff",
      "size_bytes": 48544,
      "modified_time": 1778528731.3379312
    }
  ]
}


## 6. Resolve the descriptor through Tiled

In [18]:
def get_data_with_retry(descriptor_json, timeout=30, interval=2):
    start = time.time()
    last_error = None
    while time.time() - start < timeout:
        try:
            return json.loads(tiled.get_data(descriptor_json))
        except Exception as exc:
            last_error = exc
            print("Tiled lookup not ready yet:", exc)
            time.sleep(interval)

    print("\nDescriptor diagnostics from the Tiled Tango device:")
    try:
        print(json.dumps(json.loads(tiled.inspect_descriptor(descriptor_json)), indent=2)[:8000])
    except Exception as diag_exc:
        print("Could not inspect descriptor:", diag_exc)

    raise RuntimeError(f"Could not resolve data through Tiled after {timeout}s") from last_error

data = get_data_with_retry(descriptor_json)

if isinstance(data, dict):
    summary = {k: data[k] for k in data.keys() if k != "data"}
    print(json.dumps(summary, indent=2)[:4000])
else:
    print(type(data), str(data)[:1000])


Tiled lookup not ready yet: DevFailed[
    DevError[
        desc = KeyError: "Could not resolve descriptor in Tiled. Tried: stem_image_haadf_20260511T154531_548f5184: 'stem_image_haadf_20260511T154531_548f5184'; stem_image_haadf_20260511T154531_548f5184.tiff: 'stem_image_haadf_20260511T154531_548f5184.tiff'"
        origin = Traceback (most recent call last):
              File "/Users/austin/Documents/GitHub/asyncroscopy/.venv/lib/python3.12/site-packages/tango/server.py", line 1790, in wrapped_command_method
                return get_worker().execute(cmd_method, *args, **kwargs)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              File "/Users/austin/Documents/GitHub/asyncroscopy/.venv/lib/python3.12/site-packages/tango/green.py", line 110, in execute
                return fn(*args, **kwargs)
                       ^^^^^^^^^^^^^^^^^^^
              File "/Users/austin/Documents/GitHub/asyncroscopy/asyncroscopy/Tiled.py", line 160, in get_data
    

KeyboardInterrupt: 

## 7. Plot if Tiled returned an array

In [ ]:
if isinstance(data, dict) and data.get("type") == "ndarray":
    image = np.asarray(data["data"], dtype=np.dtype(data["dtype"]))
    image = image.reshape(data["shape"])

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="gray", interpolation="none")
    ax.set_title(f"HAADF from Tiled - {descriptor.get('format')}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Tiled did not return a direct ndarray. Inspect data and traverse the returned node/path as needed.")
